In [ ]:
-- トラッキング用のクエリタグを設定
ALTER SESSION SET query_tag = '{"origin":"sf_sit-is","name":"quantitative_research_with_ai_functions_and_cortex_code","version":{"major":1,"minor":0},"attributes":{"is_quickstart":1,"source":"notebook"}}';

# 金融サービス向け Cortex

Snowflake は金融サービスのお客様に信頼されるデータプラットフォームとして、最新の AI 機能を活用してワークフローをさらに強化する方法をご紹介します。このノートブックでは以下を行います：

1. **アナリストセンチメントの抽出** - Cortex AI SQL (`AI_COMPLETE`) を使用して決算説明会のトランスクリプトを処理
2. **Cortex Search の作成** - センチメントインサイトに対するセマンティック検索を有効化
3. **Semantic View の作成** - Cortex Analyst による自然言語クエリを有効化

## 📝 Cortex AISQL: 非構造化データを構造化データに変換


**ダウ・ジョーンズ30社の決算説明会**（2024年以降）を対象とした **AI ベースのセンチメントデータセット**。
- **データソース**: `COMPANY_EVENT_TRANSCRIPT_ATTRIBUTES_V2` からの注釈付き Q&A トランスクリプト  
- **AI スコアリング**: Claude (`claude-4-sonnet`) が**アナリストの質問・トーンのみ**を分析し、経営陣の準備された発言は無視  
- **出力**: 以下を含む JSON  
  - `sentiment_score` (1–10 スケール)  
  - `reason` (簡潔な理由)  
  - `analyst_count` (ユニークなアナリスト数)  

In [ ]:
-- ダウ・ジョーンズ30社のアナリストセンチメントを作成 
-- 注意: タイムスタンプは UTC

create or replace table ai_transcripts_analysts_sentiments AS (
WITH 
ai_analysis AS (
    SELECT
        primary_ticker,
        event_timestamp,
        event_type,
        created_at,
        ai_complete(
            'claude-4-sonnet', -- 現時点では Claude を使用 
            CONCAT_WS('\n',
                'You are analyzing sell side and buy side analysts sentiment in a public company''s earnings call transcript.',
                'Focus ONLY on analyst questions, tone, and reactions to management responses.- Ignore management's prepared remarks unless directly referenced by analysts. - Pay attention to how analysts compare results to prior earnings calls and to market expectations.',
                'Use a CONSISTENT STANDARD across all earnings calls, regardless of sector, size, or company specifics.',
                'Evaluate (internally, equal weight) the following aspects:',
                'a) Guidance/Financial Outlook (revenue growth, earnings, margins, bookings, cash flow)',
                'b) Strategy/Product/Innovation (new initiatives, products, technologies, or business models)',
                'c) Competitive/Market Positioning (market share, competition, regulation, macro/sector trends)',
                'd) Management Execution & Credibility (track record, transparency, quality of responses, consistency with prior guidance)',
                'Compute ONE overall analyst sentiment score on a 1-10 scale, where:',
                '1 = Extremely pessimistic/concerned, 5 = Neutral/mixed, 10 = Extremely optimistic/bullish.',
                'Return your response as valid JSON with this exact format:',
                '{"score": <integer 1-10>, "reason": <brief explanation covering the four aspects>, "analyst_count": <integer number of unique analysts>}',
                'Transcript:',
                transcript
            )
        ) AS ai_response
    FROM unique_transcripts
)

SELECT
    primary_ticker,
    event_timestamp,
    event_type,
    created_at,
    /* JSON を安全にパース。パースに失敗した場合は NULL */
    (TRY_PARSE_JSON(ai_response):score)::INT      AS sentiment_score,
    (TRY_PARSE_JSON(ai_response):analyst_count)::INT AS unique_analyst_count,
    (TRY_PARSE_JSON(ai_response):reason)::STRING  AS sentiment_reason
FROM ai_analysis
-- where unique_analyst_count > 1 -- 企業側の発言者のみのデータを除外
order by primary_ticker, event_timestamp
);

select * from ai_transcripts_analysts_sentiments;

## ✅ ノートブック完了！

Cortex AI Functions を使用して、決算説明会のトランスクリプトから**アナリストセンチメントの抽出**に成功しました。

### 次のステップ
1. **TRAIN_ML_MODELS** ノートブックを開いて、ML 予測モデルをトレーニングして登録
2. **scripts/create_cortex_components.sql** を実行して、Cortex Search、Semantic View、Agent を作成
3. **Snowflake Intelligence** でエージェントをテスト